### Imports
Import Python modules for executing the notebook.
 - Pandas is used for building and handling dataframes.
 - Scrapbook is used for recording data for the Notebook Execution Service.
 - SystemsApi is an NI provided package for communicating with the SystemLink Systems service.

In [ ]:
import pandas as pd
import scrapbook as sb

from systemlink.clients.nisysmgmt.api.systems_api import SystemsApi
from systemlink.clients.nisysmgmt.models.query_systems_request import QuerySystemsRequest

### Parameters
 - `group_by`: The property by which data is grouped. For the data to appear as a column in the Systems Grid, we must support 'System' here.
 - `package`: The Package Name of the software to display the version of (e.g. ni-daqmx).
 - `systems_filter`: A filter specifying which systems to query. An empty filter matches all systems.

In [ ]:
group_by = "System"
package = "ni-daqmx"
systems_filter = ""

### Query for Systems with the specified package and get the Package Version on each system

In [ ]:
api = SystemsApi()

projection = f"new(id, packages.data[\"{package}\"].displayversion, packages.data[\"{package}\"].version)"
filter = (systems_filter or "!string.IsNullOrEmpty(id)") + f" && packages.data.keys.Contains(\"{package}\")"

query_sys_request = QuerySystemsRequest(skip=0, projection=projection, filter=filter)
query_result = api.get_systems_by_query(query=query_sys_request)
data = await query_result

### Extract Package data from query results and create pandas dataframe

In [ ]:
pkg_version = { item['id'] : item['displayversion'] for item in data.data }
df = pd.DataFrame.from_dict(pkg_version, orient='index', columns=['Package Version'])
df

### Convert dataframe to result format that the Systems Grid can interpret

In [ ]:
df_dict = {
    'columns': ['minion id', 'package version'],
    'values': df.reset_index().values.tolist()
}

result = [{
    "display_name": "Package Version",
    "id": "package_version",
    "type": "data_frame",
    "data": df_dict
}]

sb.glue('result', result)

### View the output of this report in the Systems Grid
1. In SystemLink, navigate to Utilities -> Jupyter, and create a folder named 'reports' if it does not already exist
1. Upload this notebook to the reports folder
1. From the Systems page, press the edit grid button in the upper-right section of the grid area
1. Press the '+ ADD' button to add a new column, and select 'Notebook' as the data source
1. Select this report in the Path field
1. In the Output field, select 'Package Version'
1. In the Package field, enter the name of the package (default: ni-daqmx)
1. Select an appropriate Update interval for your needs (5 min to 24 hours)
1. Enter an appropriate Column name (e.g. 'DAQmx Version') and press Done